In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset, ConcatDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from tqdm import tqdm
import pandas as pd
import glob
import os
import random

# -------------------------------------------------
# 1. אוגמנטציות מתקדמות וגיוון נתונים
# -------------------------------------------------

def add_gaussian_noise(signal, noise_range=(0.02, 0.1)):
    """הוספת רעש גאוסי באמפליטודה משתנה"""
    noise_level = np.random.uniform(*noise_range)
    noise = np.random.normal(0, noise_level, signal.shape)
    return signal + noise

def time_shift(signal, max_shift_ratio=0.2):
    """הזזה רנדומלית בזמן (כאחוז מהאורך)"""
    max_shift = int(signal.shape[0] * max_shift_ratio)
    shift = np.random.randint(-max_shift, max_shift)
    shifted = np.zeros_like(signal)
    
    if shift > 0:
        shifted[shift:, :] = signal[:-shift, :]
    elif shift < 0:
        shifted[:shift, :] = signal[-shift:, :]
    else:
        shifted = signal
    
    return shifted

def amplitude_scaling(signal, scaling_range=(0.5, 2.0)):
    """שינוי סקאלה של האמפליטודה"""
    scaling_factor = np.random.uniform(*scaling_range)
    return signal * scaling_factor

def time_warping(signal, strength_range=(0.1, 0.3)):
    """עיוות בציר הזמן"""
    strength = np.random.uniform(*strength_range)
    window_size, channels = signal.shape
    
    original_time = np.arange(window_size)
    
    random_warping = np.random.uniform(-strength, strength, size=3)
    warped_time = original_time.copy().astype(float)
    for i, coeff in enumerate(random_warping):
        warped_time += coeff * (original_time**(i+1)) / (window_size**i)
    
    warped_time = np.clip(warped_time, 0, window_size-1)
    
    warped_signal = np.zeros_like(signal)
    for c in range(channels):
        from scipy.interpolate import interp1d
        interpolator = interp1d(original_time, signal[:, c], 
                              kind='linear', bounds_error=False, fill_value=0)
        warped_signal[:, c] = interpolator(warped_time)
    
    return warped_signal

def frequency_masking(signal, mask_width_range=(10, 100)):
    """מיסוך תדרים מסוימים"""
    window_size, channels = signal.shape
    masked_signal = signal.copy()
    
    mask_width = np.random.randint(*mask_width_range)
    mask_start = np.random.randint(0, window_size//2 - mask_width)
    
    for c in range(channels):
        fft = np.fft.rfft(signal[:, c])
        fft[mask_start:mask_start+mask_width] = 0
        masked_signal[:, c] = np.fft.irfft(fft, n=window_size)
    
    return masked_signal

def add_sinusoidal_noise(signal, freq_range=(5, 50), amp_range=(0.05, 0.15)):
    """הוספת רעש סינוסואידלי (כמו הפרעות חשמל)"""
    window_size, channels = signal.shape
    time = np.arange(window_size)
    
    freq = np.random.uniform(*freq_range)
    amp = np.random.uniform(*amp_range)
    
    # יצירת הרעש הסינוסואידלי
    sin_noise = amp * np.sin(2 * np.pi * freq * time / window_size)
    
    # הוספה לכל הערוצים
    noisy_signal = signal.copy()
    for c in range(channels):
        noisy_signal[:, c] = signal[:, c] + sin_noise
    
    return noisy_signal

def simulate_sensor_variation(signal, variation_range=(0.8, 1.2)):
    """סימולציה של וריאציות בין חיישנים שונים"""
    window_size, channels = signal.shape
    modified_signal = signal.copy()
    
    # מקדם וריאציה שונה לכל ערוץ
    for c in range(channels):
        channel_factor = np.random.uniform(*variation_range)
        modified_signal[:, c] = signal[:, c] * channel_factor
    
    return modified_signal

def elastic_distortion(signal, distortion_range=(0.05, 0.2)):
    """עיוות אלסטי - משנה את האות בצורה לא לינארית"""
    window_size, channels = signal.shape
    distorted_signal = signal.copy()
    
    distortion_strength = np.random.uniform(*distortion_range)
    distortion = np.cumsum(np.random.normal(0, distortion_strength, window_size))
    distortion = distortion / np.max(np.abs(distortion)) * window_size * 0.1
    
    for c in range(channels):
        from scipy.interpolate import interp1d
        x = np.arange(window_size)
        x_distorted = x + distortion
        x_distorted = np.clip(x_distorted, 0, window_size-1)
        
        interpolator = interp1d(x, signal[:, c], kind='cubic', bounds_error=False, fill_value=0)
        distorted_signal[:, c] = interpolator(x_distorted)
    
    return distorted_signal

def frequency_shift(signal, shift_range=(-20, 20)):
    """הזזת האות בתחום התדר"""
    window_size, channels = signal.shape
    shifted_signal = signal.copy()
    
    # הזזת תדר שונה לכל ערוץ
    for c in range(channels):
        shift = np.random.uniform(*shift_range)
        
        # FFT
        fft = np.fft.rfft(signal[:, c])
        freq_shift = int(shift)
        
        # הזזה בתחום התדר
        if freq_shift > 0:
            fft[freq_shift:] = fft[:-freq_shift]
            fft[:freq_shift] = 0
        elif freq_shift < 0:
            fft[:freq_shift] = fft[-freq_shift:]
            fft[freq_shift:] = 0
        
        # חזרה לתחום הזמן
        shifted_signal[:, c] = np.fft.irfft(fft, n=window_size)
    
    return shifted_signal

def mix_augmented_samples(signal, other_signals, ratio_range=(0.1, 0.3)):
    """ערבוב של האות עם אותות אחרים"""
    if not other_signals:
        return signal
    
    mixed_signal = signal.copy()
    mix_ratio = np.random.uniform(*ratio_range)
    
    # בחירה אקראית של אות אחר
    other_signal = random.choice(other_signals)
    
    # ערבוב האותות
    mixed_signal = (1 - mix_ratio) * signal + mix_ratio * other_signal
    
    return mixed_signal

# פונקציה מרכזית ליצירת אוגמנטציות מגוונות
def create_augmented_dataset(windows_list, labels_list, augment_factor=5, cross_class_mixing=True):
    """
    יצירת מאגר נתונים מורחב עם אוגמנטציות מגוונות וקומבינציות שלהן.
    
    פרמטרים:
    - windows_list: רשימת מערכים מקוריים (כל אחד הוא קבוצת חלונות של מחלקה אחת)
    - labels_list: רשימת תוויות מתאימה
    - augment_factor: כמה דוגמאות מורחבות ליצור לכל דוגמה מקורית
    - cross_class_mixing: האם לערבב דוגמאות ממחלקות שונות
    
    מחזיר:
    - חלונות ותוויות מוגדלים
    """
    augmented_windows = []
    augmented_labels = []
    
    # קודם כל, העתק את כל הנתונים המקוריים
    for class_windows, class_labels in zip(windows_list, labels_list):
        augmented_windows.extend(class_windows)
        augmented_labels.extend(class_labels)
    
    print(f"מספר דוגמאות מקורי: {len(augmented_windows)}")
    
    # טכניקות אוגמנטציה בסיסיות
    basic_augmentations = [
        add_gaussian_noise,
        time_shift,
        amplitude_scaling,
        time_warping,
        frequency_masking
    ]
    
    # טכניקות אוגמנטציה מתקדמות
    advanced_augmentations = [
        add_sinusoidal_noise,
        simulate_sensor_variation,
        elastic_distortion,
        frequency_shift
    ]
    
    # כל רשימת החלונות שטוחה (ללא חלוקה למחלקות)
    flat_windows = []
    for class_windows in windows_list:
        flat_windows.extend(class_windows)
    
    # יצירת אוגמנטציות
    for class_idx, (class_windows, class_labels) in enumerate(zip(windows_list, labels_list)):
        print(f"מייצר אוגמנטציות למחלקה {class_idx}...")
        
        for i, (window, label) in enumerate(tqdm(zip(class_windows, class_labels), total=len(class_windows))):
            # חלונות אחרים עבור מיקסים אפשריים (לא כולל את החלון הנוכחי)
            other_windows = [w for j, w in enumerate(flat_windows) if j != i + sum(len(w) for w in windows_list[:class_idx])]
            
            for _ in range(augment_factor):
                # יצירת דוגמה חדשה עם אוגמנטציה
                augmented_window = window.copy()
                
                # בחירה אקראית של מספר אוגמנטציות להחלה (1-3)
                num_augmentations = np.random.randint(1, 4)
                
                # שילוב של אוגמנטציות בסיסיות ומתקדמות
                all_augmentations = basic_augmentations + advanced_augmentations
                selected_augmentations = np.random.choice(all_augmentations, size=num_augmentations, replace=False)
                
                # החלת האוגמנטציות הנבחרות
                for aug_func in selected_augmentations:
                    augmented_window = aug_func(augmented_window)
                
                # אפשרות לערבוב בין דוגמאות (cross-class mixing)
                if cross_class_mixing and np.random.random() < 0.2 and other_windows:
                    augmented_window = mix_augmented_samples(augmented_window, other_windows)
                
                # הוספה למאגר המוגדל
                augmented_windows.append(augmented_window)
                augmented_labels.append(label)
    
    print(f"מספר דוגמאות לאחר הגדלה: {len(augmented_windows)}")
    return augmented_windows, augmented_labels

# -------------------------------------------------
# 2. ארכיטקטורת מודל חזקה עם מנגנוני עמידות
# -------------------------------------------------

class RobustGeophoneCNN(nn.Module):
    def __init__(self, in_channels, num_classes=3):
        super(RobustGeophoneCNN, self).__init__()
        
        # יכולות ליניאריות
        self.conv_layers = nn.ModuleList([
            # בלוק 1
            nn.Conv1d(in_channels, 32, kernel_size=9, padding=4),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),
            
            # בלוק 2
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(2),
            nn.Dropout(0.3),
            
            # בלוק 3
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(2),
            nn.Dropout(0.3),
            
            # בלוק 4
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(2),
            nn.Dropout(0.4),
        ])
        
        # יכולות גלובליות - זיהוי מאפיינים גלובליים באות
        window_size = 2000
        reduced_size = window_size // 16  # אחרי 4 MaxPool
        
        # שכבות fully connected
        self.fc_layers = nn.ModuleList([
            nn.Linear(256 * reduced_size, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.5),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.5),
            
            nn.Linear(256, num_classes)
        ])
        
        # מדגישים טווחי תדרים בעלי חשיבות
        self.frequency_attention = FrequencyAttention(256)
    
    def forward(self, x):
        # העברת השכבות הקונבולוציוניות
        for i, layer in enumerate(self.conv_layers):
            x = layer(x)
            
            # הפעלת מנגנון תשומת לב לתדרים אחרי בלוק 3
            if i == 11:  # אחרי הבלוק השלישי
                x = self.frequency_attention(x)
        
        # שינוי צורה
        x = x.view(x.size(0), -1)
        
        # שכבות fc
        for layer in self.fc_layers:
            x = layer(x)
        
        return x

# מודול תשומת לב לתדרים
class FrequencyAttention(nn.Module):
    def __init__(self, channels):
        super(FrequencyAttention, self).__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, channels // 4, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv1d(channels // 4, channels, kernel_size=1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        attention_weights = self.attention(x)
        return x * attention_weights

# -------------------------------------------------
# 3. אסטרטגיות אימון מתקדמות
# -------------------------------------------------

def train_with_advanced_strategies(model, train_loader, val_loader, test_loader, 
                                 epochs=100, lr=0.001, device='cuda' if torch.cuda.is_available() else 'cpu',
                                 patience=15, class_weights=None):
    """
    אימון המודל עם אסטרטגיות מתקדמות:
    - רגולריזציה חזקה
    - מדדי הערכה מרובים
    - שיפור הדרגתי של הביצועים
    - אופטימיזציה של היפר-פרמטרים 
    - early stopping
    """
    print(f"אימון על {device}...")
    model.to(device)
    
    # קריטריון עם שקלול מחלקות (אם סופקו משקלות)
    if class_weights is not None:
        class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    else:
        criterion = nn.CrossEntropyLoss()
    
    # אופטימייזר עם רגולריזציית L2
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # שימוש ב-scheduler להקטנת קצב הלמידה
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)
    
    # ערכים למעקב
    best_val_loss = float('inf')
    best_val_acc = 0.0
    best_model_state = None
    epochs_no_improve = 0
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'lr': []
    }
    
    for epoch in range(1, epochs + 1):
        # אימון
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (inputs, targets) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")):
            inputs, targets = inputs.to(device), targets.to(device)
            inputs = inputs.permute(0, 2, 1)  # לפורמט (batch, channels, time)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # רגולריזציה לנורמות הגדולות מדי, למניעת exploding gradients
            l2_reg = 0.0
            for param in model.parameters():
                l2_reg += torch.norm(param)
            loss += 1e-5 * l2_reg
            
            loss.backward()
            
            # חיתוך גרדיאנטים למניעת התפוצצויות
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item() * targets.size(0)
            _, predicted = outputs.max(1)
            train_total += targets.size(0)
            train_correct += predicted.eq(targets).sum().item()
        
        train_loss = train_loss / train_total
        train_acc = train_correct / train_total
        
        # ולידציה
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                inputs = inputs.permute(0, 2, 1)
                
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item() * targets.size(0)
                _, predicted = outputs.max(1)
                val_total += targets.size(0)
                val_correct += predicted.eq(targets).sum().item()
        
        val_loss = val_loss / val_total
        val_acc = val_correct / val_total
        
        # עדכון הscheduler
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        # שמירת ערכים להיסטוריה
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        
        print(f"Epoch {epoch}/{epochs} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, "
              f"LR: {current_lr:.6f}")
        
        # בדיקת שיפור
        if val_loss < best_val_loss:
            print(f"Val loss improved from {best_val_loss:.4f} to {val_loss:.4f}")
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_model_state = model.state_dict()
            epochs_no_improve = 0
            
            # שמירת המודל
            torch.save(model.state_dict(), 'best_robust_model.pth')
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epochs")
            
            # early stopping
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch} epochs")
                break
    
    # טעינת המודל הטוב ביותר
    model.load_state_dict(best_model_state)
    
    # בדיקה על סט הבדיקה
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0
    all_targets = []
    all_predictions = []
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            inputs = inputs.permute(0, 2, 1)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            test_loss += loss.item() * targets.size(0)
            _, predicted = outputs.max(1)
            test_total += targets.size(0)
            test_correct += predicted.eq(targets).sum().item()
            
            all_targets.extend(targets.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
    
    test_loss = test_loss / test_total
    test_acc = test_correct / test_total
    
    print(f"Final Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")
    
    # חישוב מטריצת בלבול וציון F1
    from sklearn.metrics import confusion_matrix, classification_report
    cm = confusion_matrix(all_targets, all_predictions)
    report = classification_report(all_targets, all_predictions)
    
    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(report)
    
    # גרף ההתקדמות
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Validation')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 3, 2)
    plt.plot(history['train_acc'], label='Train')
    plt.plot(history['val_acc'], label='Validation')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 3, 3)
    plt.plot(history['lr'])
    plt.title('Learning Rate')
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.show()
    
    return model, test_acc, history

# -------------------------------------------------
# 4. זיהוי מגוון מאגרי נתונים והערכה
# -------------------------------------------------

def evaluate_on_diverse_datasets(model, original_data, new_data_sources, class_names, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """
    בדיקת המודל על מגוון מקורות נתונים שונים
    
    פרמטרים:
    - model: המודל המאומן
    - original_data: הנתונים המקוריים
    - new_data_sources: רשימת מקורות נתונים חדשים
    - class_names: שמות המחלקות
    
    מחזיר:
    - תוצאות הערכה למקורות השונים
    """
    model.to(device)
    model.eval()
    
    results = {}
    
    # הגדרת עיבודי מעטפת
    def preprocess_data(data, target_channels):
        # pad channels if needed
        if data.shape[1] < target_channels:
            padding = np.zeros((data.shape[0], target_channels - data.shape[1]))
            data = np.hstack((data, padding))
        elif data.shape[1] > target_channels:
            data = data[:, :target_channels]
        
        # apply bandpass filter
        fs = 1000.0
        lowcut, highcut = 1.0, 100.0
        b, a = butter(4, [lowcut/(fs/2), highcut/(fs/2)], btype='band')
        filtered_data = filtfilt(b, a, data, axis=0)
        
        return filtered_data
    
    # בדיקה על הנתונים המקוריים
    print("בודק ביצועים על הנתונים המקוריים...")
    original_accuracy = evaluate_dataset(model, original_data['test_loader'], device)
    results['original'] = original_accuracy
    
    # בדיקה על כל מקור נתונים חדש
    for source_name, source_data in new_data_sources.items():
        print(f"בודק ביצועים על {source_name}...")
        if 'loader' in source_data:
            accuracy = evaluate_dataset(model, source_data['loader'], device)
        else:
            # אם אין loader מוכן, יצירת loader מנתונים גולמיים
            data = source_data['data']
            labels = source_data['labels']
            
            # עיבוד אם צריך
            processed_data = []
            for sample in data:
                processed = preprocess_data(sample, original_data['channels'])
                processed_data.append(processed)
            
            # נרמול לפי סטטיסטיקות האימון המקוריות
            normalized_data = [(d - original_data['train_mean']) / original_data['train_std'] for d in processed_data]
            
            # יצירת loader
            tensor_data = torch.tensor(np.stack(normalized_data), dtype=torch.float32)
            tensor_labels = torch.tensor(labels, dtype=torch.long)
            dataset = TensorDataset(tensor_data, tensor_labels)
            loader = DataLoader(dataset, batch_size=16, shuffle=False)
            
            accuracy = evaluate_dataset(model, loader, device)
        
        results[source_name] = accuracy
    
    # הצגת תוצאות
    print("\nSummary of model performance across datasets:")
    for dataset_name, acc in results.items():
        print(f"{dataset_name}: {acc:.2f}%")
    
    # גרף תוצאות
    plt.figure(figsize=(10, 6))
    plt.bar(results.keys(), [acc for acc in results.values()], color='skyblue')
    plt.ylabel('Accuracy (%)')
    plt.title('Model Performance Across Different Datasets')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('cross_dataset_performance.png')
    plt.show()
    
    return results

def evaluate_dataset(model, data_loader, device):
    """בדיקת דיוק על סט נתונים ספציפי"""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            inputs = inputs.permute(0, 2, 1)
            
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return 100.0 * correct / total

# -------------------------------------------------
# 5. הפעלת התהליך המלא
# -------------------------------------------------

def train_robust_model(original_man_wins, original_car_wins, original_noise_wins, new_data=None):
    """
    אימון מודל עמיד עם יכולת הכללה גבוהה
    
    פרמטרים:
    - original_man_wins: חלונות צעדים מקוריים
    - original_car_wins: חלונות רכב מקוריים
    - original_noise_wins: חלונות רעש מקוריים
    - new_data: נתונים חדשים לשילוב (אופציונלי)
    
    מחזיר:
    - המודל המאומן
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    # 1. יצירת אוגמנטציות והרחבת מאגר הנתונים
    print("יוצר מאגר נתונים מורחב עם אוגמנטציות...")
    
    # הכנת הנתונים המקוריים
    original_windows_list = [original_man_wins, original_car_wins, original_noise_wins]
    original_labels_list = [
        [0] * len(original_man_wins),  # Steps
        [1] * len(original_car_wins),  # Vehicle
        [2] * len(original_noise_wins)  # Noise
    ]
    
    # שילוב נתונים חדשים (אם יש)
    if new_data:
        for data_name, data_dict in new_data.items():
            if 'windows' in data_dict and 'labels' in data_dict:
                print(f"משלב נתונים חדשים: {data_name}")
                original_windows_list.append(data_dict['windows'])
                original_labels_list.append(data_dict['labels'])
    
    # יצירת אוגמנטציות
    augmented_windows, augmented_labels = create_augmented_dataset(
        original_windows_list, 
        original_labels_list,
        augment_factor=5,  # כמה דוגמאות חדשות ליצור עבור כל דוגמה מקורית
        cross_class_mixing=True  # לערבב בין מחלקות שונות
    )
    
    # 2. חלוקה לסטים אימון/ולידציה/בדיקה
    print("מחלק את הנתונים לסטים אימון, ולידציה ובדיקה...")
    
    # המרה לנומפיי
    X = np.stack(augmented_windows)
    y = np.array(augmented_labels)
    
    # חלוקה ראשונית לאימון+ולידציה ובדיקה (80% / 20%)
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # חלוקת סט האימון לאימון וולידציה (75% / 25% מהסט המקורי = 60% / 20%)
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
    )
    
    print(f"גודל סט אימון: {len(X_train)}, ולידציה: {len(X_val)}, בדיקה: {len(X_test)}")
    
    # חישוב סטטיסטיקות נרמול מסט האימון
    train_mean = X_train.mean(axis=(0, 1))
    train_std = X_train.std(axis=(0, 1)) + 1e-6  # הוספת קבוע קטן למניעת חלוקה באפס
    
    # נרמול הנתונים
    X_train_norm = (X_train - train_mean) / train_std
    X_val_norm = (X_val - train_mean) / train_std
    X_test_norm = (X_test - train_mean) / train_std
    
    # יצירת טנסורים
    X_train_tensor = torch.tensor(X_train_norm, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_val_tensor = torch.tensor(X_val_norm, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test_norm, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)
    
    # יצירת סטי נתונים ומטעינים
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32)
    test_loader = DataLoader(test_dataset, batch_size=32)
    
    # 3. חישוב משקלות מחלקה לטיפול בחוסר איזון
    class_counts = np.bincount(y_train)
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum() * len(class_weights)  # נרמול
    print(f"משקלות מחלקה: {class_weights}")
    
    # 4. יצירת המודל
    print("יוצר מודל חזק...")
    in_channels = X_train.shape[2]
    model = RobustGeophoneCNN(in_channels=in_channels, num_classes=3)
    
    # 5. אימון המודל
    print("מתחיל אימון...")
    trained_model, test_accuracy, history = train_with_advanced_strategies(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        epochs=100,  # מספר אפוקות מקסימלי
        lr=0.001,   # קצב למידה התחלתי
        device=device,
        patience=15,  # early stopping
        class_weights=class_weights
    )
    
    # 6. שמירת המודל המאומן
    torch.save({
        'model_state_dict': trained_model.state_dict(),
        'train_mean': train_mean,
        'train_std': train_std,
        'in_channels': in_channels,
        'history': history,
        'test_accuracy': test_accuracy
    }, 'robust_geophone_model.pth')
    
    print(f"המודל נשמר לקובץ 'robust_geophone_model.pth' עם דיוק בדיקה של {test_accuracy:.2f}%")
    
    # 7. החזרת המודל ופרטים חשובים
    return {
        'model': trained_model,
        'train_mean': train_mean,
        'train_std': train_std,
        'test_accuracy': test_accuracy,
        'channels': in_channels,
        'train_loader': train_loader,
        'val_loader': val_loader,
        'test_loader': test_loader
    }

# דוגמה לשימוש:
result = train_robust_model(man_wins, car_wins, noise_wins, new_data={'new_steps': {'windows': new_windows, 'labels': new_labels}})